<a href="https://colab.research.google.com/github/JunyiJ/DeepGenerativeModels/blob/main/Practice_of_nearest_neighbor_as_a_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nearest Neighbor as a Neural Network

**Company:** `oai` · **Time box:** 35–45 minutes · **Python + NumPy**

Reconstructed from user-provided interview notes; not an official interview transcript.

Work through the notebook in order. Fill in the implementation cells and written explanations, then run the checks. Stubs intentionally raise `NotImplementedError` until you implement them. No solutions are included.

In Colab, save a copy to Drive to keep your work. A CPU runtime is sufficient.


## Problem

You are given:

- `X_train`: a float array of shape `(n, d)` containing training points.
- `y_train`: an integer array of shape `(n,)` containing labels. Labels may repeat and need not be consecutive.
- `X_query`: a float array of shape `(m, d)` containing query points.

Assume inputs are finite, `n > 0`, and `d > 0`.

For each query, return the label of its nearest training point under Euclidean distance. If multiple training points are equally close, choose the one with the **smallest training index**.



In [1]:
import numpy as np

print("NumPy version:", np.__version__)


NumPy version: 2.1.3


## Part 1 — Vectorized nearest neighbor

Implement:

Use the implementation cell below.

Requirements:

- Use vectorized NumPy operations without loops over individual points.
- Avoid constructing an intermediate array of shape `(m, n, d)`.
- Explain the shapes of your intermediate arrays.
- State time and auxiliary-space complexity.



In [41]:
def predict_nearest_neighbor(X_train, y_train, X_query):
    """Return predicted labels with shape (m,); break ties by training index."""
    # (m, n)
    A = np.sum(X_query * X_query, axis=1, keepdims=True)
    B = np.sum(X_train * X_train, axis=1)
    C = X_query @ (X_train.T)
    distances =  A - 2 * C + B.T
    # (m, 1)
    min_indices = np.argmin(distances, axis=1)
    return y_train[min_indices]



### Explain your approach

- Intermediate arrays and their shapes: distance: (m, n)
- Time complexity: matrix calculation distances: (m * n * d)
- Auxiliary-space complexity: mn
- Tie-breaking behavior: np.argmin return first index


In [42]:
X_train = np.array([[0., 0.], [3., 0.], [0., 3.]])
y_train = np.array([4, 8, 8])
X_query = np.array([[2.6, 0.2], [0., 0.], [1.5, 0.]])
expected = np.array([8, 4, 4])

np.testing.assert_array_equal(
    predict_nearest_neighbor(X_train, y_train, X_query), expected
)
print("Part 1 example passed.")


Part 1 example passed.


## Part 2 — A single affine layer and an activation

Construct weights and biases directly from the training points, without training. Use one affine transformation followed by softmax and a label lookup to reproduce Part 1.

- Choose and document your weight orientation and all tensor shapes.
- Derive why the selected training point matches Part 1.
- Implement numerically stable softmax over each query's training-point scores.
- Preserve the smallest-index tie rule.
- Handle repeated and nonconsecutive labels.

The extra `softmax` helper makes numerical stability independently testable.


In [79]:
def build_nn_parameters(X_train):
    """Return W and b constructed directly from the training points."""
    # (n, d)
    W = 2 * X_train
    # (n)
    b = np.sum(X_train * X_train, axis=1)
    return W.T, -b


def softmax(logits):
    """Return row-wise probabilities for logits with shape (m, n)."""
    max = np.max(logits, axis=1, keepdims=True)
    exp = np.exp(logits - max)
    sum = np.sum(exp, axis=1, keepdims=True)
    return exp / sum


def predict_with_layer(X_query, y_train, W, b):
    """Apply the affine layer, softmax, and label lookup; return shape (m,)."""
    X1 = X_query @ W + b - np.sum(X_query * X_query, axis=1, keepdims=True)
    X2 = softmax(X1)
    return y_train[np.argmax(X2, axis=1)]


### Derivation and shapes

_Write your derivation here before running the checks._

- `W` shape: _Your answer_
- `b` shape: _Your answer_
- Affine output shape: _Your answer_
- Why predictions agree with Part 1: _Your answer_
- Mapping training-point outputs to labels: _Your answer_


In [76]:
W, b = build_nn_parameters(X_train)
np.testing.assert_array_equal(
    predict_with_layer(X_query, y_train, W, b), expected
)
print("Part 2 example passed.")


Part 2 example passed.


## Behavioral checks

Run these after implementing both parts. They cover exact ties, duplicates with different labels, repeated labels, one training point, empty query batches, and random small inputs.

The direct-distance computation below is a small test oracle. The exercise implementations must still satisfy the vectorization and memory requirements. These checks verify outputs, not whether you followed every implementation constraint.


In [77]:
def check_case(train, labels, queries, expected_labels):
    first = predict_nearest_neighbor(train, labels, queries)
    weights, bias = build_nn_parameters(train)
    second = predict_with_layer(queries, labels, weights, bias)
    assert np.asarray(first).shape == (len(queries),)
    assert np.asarray(second).shape == (len(queries),)
    np.testing.assert_array_equal(first, expected_labels)
    np.testing.assert_array_equal(second, expected_labels)


# Ties must use training order, not the numerical order of labels.
check_case(np.array([[-1.], [1.]]), np.array([99, -5]),
           np.array([[0.], [1.]]), np.array([99, -5]))

# Duplicate points with different labels; multiple points share a label.
check_case(np.array([[0., 0.], [0., 0.], [2., 0.], [0., 2.]]),
           np.array([42, -7, 42, 9]),
           np.array([[0., 0.], [2., 0.], [0., 2.]]), np.array([42, 42, 9]))

# One training point.
check_case(np.array([[2., -1.]]), np.array([-12]),
           np.array([[0., 0.], [100., -100.]]), np.array([-12, -12]))

# Empty query batch.
check_case(np.array([[0., 0.]]), np.array([7]),
           np.empty((0, 2)), np.array([], dtype=int))

# Independent, intentionally simple oracle for small random cases.
rng = np.random.default_rng(42)
for n, m, d in [(7, 5, 3), (1, 4, 2), (13, 8, 6)]:
    train = rng.normal(size=(n, d))
    queries = rng.normal(size=(m, d))
    labels = rng.choice([-11, 4, 99], size=n)
    expected_labels = np.array([
        labels[np.argmin(np.sum((train - query) ** 2, axis=1))]
        for query in queries
    ])
    check_case(train, labels, queries, expected_labels)

print("All prediction checks passed.")


All prediction checks passed.


In [78]:
# Large magnitudes should not overflow; probabilities normalize per row.
logits = np.array([[10000., 10001., 9999.], [-10000., -10001., -9999.]])
with np.errstate(over="raise", invalid="raise", divide="raise"):
    probs = softmax(logits)
assert probs.shape == logits.shape
assert np.isfinite(probs).all()
assert (probs >= 0).all()
np.testing.assert_allclose(probs.sum(axis=1), np.ones(2))
np.testing.assert_array_equal(probs.argmax(axis=1), np.array([1, 2]))
np.testing.assert_allclose(softmax(np.zeros((2, 3))), np.full((2, 3), 1 / 3))
np.testing.assert_allclose(softmax(logits + 50000), probs)
print("Softmax checks passed.")


Softmax checks passed.


## Follow-ups

1. Can you avoid taking square roots? Why?
2. Is softmax necessary to obtain the predicted label?
3. Does aggregating softmax probabilities by class always reproduce 1-NN? Justify your answer.
4. What would you change if the `(m, n)` matrix did not fit in memory?
5. How would you adapt this approach to cosine similarity? What edge case requires special handling?
6. Which tests would you write to check that both implementations agree?
7. Can floating-point arithmetic affect equivalence or tie-breaking? How would you investigate discrepancies?


Write your answers below.


Yes, because square root is strictly increasing on nonnegative values. Minimizing squared Euclidean distance selects the same point as minimizing Euclidean distance:
\[
\arg\min_i \sqrt{D_i}=\arg\min_i D_i.
\]
Your monotonicity reasoning is correct, but log probabilities aren’t involved here.

Softmax is unnecessary for the predicted label. Use argmax directly on the affine scores:
\[
z_i=2q^\top x_i-\|x_i\|^2.
\]
Softmax preserves their ordering in exact arithmetic. Then look up y_train at the winning training-point index.

No. Aggregating probabilities by class can change the prediction. Each softmax output corresponds to a training point. Suppose:
Training point	Label	Softmax probability
0	A	0.40
1	B	0.35
2	B	0.25


1-NN picks point 0, so it predicts A. Summing probabilities by label gives A = 0.40 and B = 0.60, so class aggregation predicts B. Several farther points can collectively outweigh the single nearest point.

Process queries in batches. Your proposal works: computing b queries at a time reduces distance-matrix memory from O(mn) to O(bn). If even one query against all training points is too large, also batch training points. Keep each query’s best distance and global training index across batches. Preserve the smallest index on ties.

Normalize vectors and maximize their dot product.
\[
\cos(q,x)=\frac{q^\top x}{\|q\|\|x\|}.
\]
Normalize both training and query vectors to unit length, then use X_query_normalized @ X_train_normalized.T and take argmax. For the affine layer, use normalized training vectors as weights and zero bias. Zero-length vectors need an explicit policy, because cosine similarity is undefined for them.

Compare both implementations against independent expected answers. Agreement alone is insufficient because both could share the same bug. Test:
- The provided example and small hand-calculated cases.
- Exact ties and duplicate points with different labels.
- Repeated, negative, and nonconsecutive labels.
- One training point and empty query batches.
- Different numbers of queries and training points, to catch broadcasting bugs.
- Random small inputs checked against a simple direct-distance implementation.

Yes. Mathematically equivalent formulas can differ in floating-point arithmetic. The expanded distance formula can lose precision when subtracting large, nearly equal terms. Near ties can change winners; softmax can also round distinct scores to equal probabilities.
To investigate, inspect the winning and runner-up indices, their scores, and the gap between them. Recompute using float64 and direct differences, sum((q - x)**2), as a reference. Float64 improves precision but does not give exact arithmetic. Compare numerical values with tolerances, and examine prediction disagreements individually—especially near ties.

## Practice log

- Date:
- Time spent:
- Completed without hints:
- Bugs or gaps discovered:
- What to revisit:
